In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [6]:
DATA_DIR = '../Downloads/415files/'

In [7]:
def analyze(name, X, y):
    print(f"\n--- {name} ---")
    print(f"shape: {X.shape}, target dist: {dict(pd.Series(y).value_counts())}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    gb = GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=5, learning_rate=0.1)
    gb.fit(X_train_s, y_train)
    pred = gb.predict(X_test_s)
    print(f"accuracy: {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, digits=4))
    imp = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False).head(5)
    print(f"top 5 features:\n{imp}")

In [8]:
df2 = pd.read_csv(DATA_DIR + 'dataset2.csv')
y2 = df2['phishing']
X2 = df2.drop('phishing', axis=1)
analyze('dataset2', X2, y2)


--- dataset2 ---
shape: (88647, 111), target dist: {0: np.int64(58000), 1: np.int64(30647)}
accuracy: 0.9624
              precision    recall  f1-score   support

           0     0.9729    0.9696    0.9712     11600
           1     0.9428    0.9489    0.9459      6130

    accuracy                         0.9624     17730
   macro avg     0.9579    0.9593    0.9585     17730
weighted avg     0.9625    0.9624    0.9625     17730

top 5 features:
directory_length          0.657333
time_domain_activation    0.113803
length_url                0.032748
qty_dot_domain            0.024030
asn_ip                    0.018206
dtype: float64


In [9]:
df3 = pd.read_csv(DATA_DIR + 'dataset3.csv')
df3 = df3.drop('url', axis=1)  # drop string URL column
y3 = (df3['status'] == 'phishing').astype(int)
X3 = df3.drop('status', axis=1)
analyze('dataset3', X3, y3)


--- dataset3 ---
shape: (11430, 87), target dist: {0: np.int64(5715), 1: np.int64(5715)}
accuracy: 0.9602
              precision    recall  f1-score   support

           0     0.9630    0.9571    0.9601      1143
           1     0.9574    0.9633    0.9603      1143

    accuracy                         0.9602      2286
   macro avg     0.9602    0.9602    0.9602      2286
weighted avg     0.9602    0.9602    0.9602      2286

top 5 features:
google_index     0.578925
page_rank        0.108396
nb_hyperlinks    0.081945
nb_www           0.033151
domain_age       0.021193
dtype: float64


In [10]:
df4 = pd.read_csv(DATA_DIR + 'dataset4.csv')
df4 = df4.drop(['FILENAME', 'URL', 'Domain', 'TLD', 'Title'], axis=1)  # drop string cols
df4 = df4.sample(n=30000, random_state=42)  # sample for tractable runtime
y4 = df4['label']
X4 = df4.drop('label', axis=1)
analyze('dataset4 (30K sample)', X4, y4)


--- dataset4 (30K sample) ---
shape: (30000, 50), target dist: {1: np.int64(17249), 0: np.int64(12751)}
accuracy: 1.0000
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      2550
           1     1.0000    1.0000    1.0000      3450

    accuracy                         1.0000      6000
   macro avg     1.0000    1.0000    1.0000      6000
weighted avg     1.0000    1.0000    1.0000      6000

top 5 features:
URLSimilarityIndex       9.849229e-01
LineOfCode               1.456575e-02
IsHTTPS                  5.113847e-04
URLCharProb              6.411518e-14
SpacialCharRatioInURL    4.057646e-14
dtype: float64


In [11]:
try:
    df5 = pd.read_csv(DATA_DIR + 'dataset5.csv', encoding='latin-1', on_bad_lines='skip')
    print(f"\n--- dataset5 ---")
    print(f"shape after skipping bad lines: {df5.shape}")
    print(f"string columns (should be numeric): {df5.select_dtypes(include=['object', 'string']).columns.tolist()}")
    print(f"target 'label' has {df5['label'].isna().sum()} NaN values")
    print("dataset5 requires significant cleaning before it can be modeled")
except Exception as e:
    print(f"dataset5 failed: {e}")


--- dataset5 ---
shape after skipping bad lines: (96005, 14)
string columns (should be numeric): ['domain', 'ranking', 'mld_res', 'mld.ps_res', 'jaccard_ARrd', 'jaccard_ARrem']
target 'label' has 92 NaN values
dataset5 requires significant cleaning before it can be modeled


In [12]:
df6 = pd.read_csv(DATA_DIR + 'dataset6.csv')
df6 = df6.drop('id', axis=1)
y6 = df6['CLASS_LABEL']
X6 = df6.drop('CLASS_LABEL', axis=1)
analyze('dataset6', X6, y6)


--- dataset6 ---
shape: (10000, 48), target dist: {1: np.int64(5000), 0: np.int64(5000)}
accuracy: 0.9855
              precision    recall  f1-score   support

           0     0.9850    0.9860    0.9855      1000
           1     0.9860    0.9850    0.9855      1000

    accuracy                         0.9855      2000
   macro avg     0.9855    0.9855    0.9855      2000
weighted avg     0.9855    0.9855    0.9855      2000

top 5 features:
PctExtHyperlinks                      0.375718
PctExtNullSelfRedirectHyperlinksRT    0.364981
FrequentDomainNameMismatch            0.053934
InsecureForms                         0.036971
PctNullSelfRedirectHyperlinks         0.026364
dtype: float64
